<!-- ATLAS LAUNCHER v1 · generated by tools/notebooks.py · do not edit by hand -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/main/atlas/programmes/predictive-maintenance/lessons/P03-L07-ot-network-deployment/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/main/atlas/programmes/predictive-maintenance/lessons/P03-L07-ot-network-deployment/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/main?labpath=atlas/programmes/predictive-maintenance/lessons/P03-L07-ot-network-deployment/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://github.com/codespaces/new?repo=TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

**Runs in:** Google Colab · Kaggle · Binder · GitHub Codespaces · local Jupyter.
This lesson needs a compiler (`clang` or `gcc`). The cell below installs anything missing and fetches the files
this lesson needs beside it; on a machine that already has them it does nothing at all.

In [ ]:
# --- ATLAS LAUNCHER v1 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

ATLAS_PIP = []            # (import name, pip name) for what this lesson imports
ATLAS_SIBLINGS = ["Makefile", "lesson.c"]    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# ATLAS_RAW_OVERRIDE before running this cell.
ATLAS_RAW = os.environ.get("ATLAS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/main/atlas/programmes/predictive-maintenance/lessons/P03-L07-ot-network-deployment/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    ATLAS_DIR = Path(__file__).resolve().parent
except NameError:
    ATLAS_DIR = Path.cwd()


def atlas_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in ATLAS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in ATLAS_SIBLINGS:
    if not (ATLAS_DIR / _name).exists():
        (ATLAS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(ATLAS_RAW + _name, ATLAS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel; "
                f"otherwise download it from {ATLAS_RAW + _name} and upload it."
            ) from None

print("ready on " + atlas_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END ATLAS LAUNCHER ---

# P03-L07 · Deployment on an OT network

**You will build:** the same inference path twice — once in C, in integers, inside a
40 KiB arena with no heap allocation after start-up, and once in numpy in float64 — plus
the store-and-forward buffer that carries a reading across a link outage exactly once, and
the unsupervised drift signal that is all a one-way data diode leaves you.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, no GPU, no download, **no socket**
· **Prerequisites:** T00-L01 (the tier gate), P03-L01 (alarm economics),
P03-L06 (thresholds from a distribution)

By the end you will be able to:
1. Implement a fixed-arena allocator and a fixed-point inference path in C, and run the
   whole week's inference without one byte of heap after start-up.
2. Measure the disagreement between the gateway's answer and the notebook's, separating
   what the arithmetic costs from what the register map costs.
3. Measure how many alarm decisions change when a threshold is not representable on the
   wire, and in which direction they change.
4. Implement a store-and-forward buffer that neither loses nor duplicates a reading across
   an outage, and measure how many hours of plant it holds.
5. Explain, from a measured comparison, why a one-way diode forces module 8's drift
   detection to be unsupervised, and implement the signal that survives it.

Four of the seven exercises are in **`lesson.c`**. This notebook builds it, drives it,
measures it and grades it.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import hashlib
import math
import subprocess
import sys
from pathlib import Path

import numpy as np

print("numpy", np.__version__, "· python", sys.version.split()[0])

# True in a notebook and when this file is run as a script; False when the autograder imports
# it. Every check below is called under this guard, so the cell you are sitting in reports on
# itself, while importing the lesson never runs anything.
_IS_MAIN = __name__ == "__main__"

try:
    LESSON_DIR = Path(__file__).resolve().parent
except NameError:  # a notebook has no __file__
    LESSON_DIR = Path.cwd()

C_SRC = "lesson.c"
BIN = "lesson_bin"

# Mirrored exactly from lesson.c. Both sides must build the same capture out of the same
# bytes or no comparison between them means anything.
SEED = 20260922
SEED2 = SEED ^ 0xA5A5A5A5A5A5A5A5
GAMMA = 0x9E3779B97F4A7C15

N_MACHINES = 80
N_HOURS = 168
BURST_LEN = 32
N_FRAMES = N_MACHINES * N_HOURS
FRAME_BYTES = 3 + 2 * BURST_LEN + 2
FUNC_READ_INPUT = 0x04
CORRUPT_STRIDE, CORRUPT_PHASE = 997, 13

N_HEALTHY, N_MARGINAL = 40, 24

SAMPLE_SHIFT = 8          # a raw count is engineering units x 2^8
Q_FRAC = 16               # the gateway computes in Q16.16
WIRE_FRAC = 8             # the register map carries Q8.8 in one 16-bit word
WIRE_SCALE = 1 << WIRE_FRAC

GW_ARENA_BYTES = 40960    # the gateway's entire heap
RING_CAPACITY = 3072      # records the store-and-forward buffer holds

# The operating point, and the only number in this lesson that arrived from somewhere else.
# Module 1 priced a confusion matrix and produced a cost-optimal threshold on this same health
# index; module 6 asked the same economic question of an RUL distribution and produced an
# intervention time. What crosses the firewall is a threshold on the health index, and this
# module's whole job is to get it onto a wire without quietly changing what it means.
THRESHOLD = 1.4732

DATA_NOTE = ("SYNTHETIC. The protocol capture, the fleet behind it and the outage script are "
             "all generated at run time from a counter-based stream seeded with 20260922, in "
             "this notebook and again in lesson.c. Nothing here opens a socket and nothing "
             "here is downloaded.")

_BUILD = None
_CACHE = {}


def build(verbose: bool = True):
    """Compile the C source with make. Cached: the compiler runs once per session."""
    global _BUILD
    if _BUILD is None:
        proc = subprocess.run(
            ["make", "-C", str(LESSON_DIR), f"PYTHON={sys.executable}",
             f"SRC={C_SRC}", f"BIN={BIN}"],
            capture_output=True, text=True, timeout=600)
        _BUILD = (proc.returncode == 0, (proc.stdout + proc.stderr).strip())
    ok, out = _BUILD
    if verbose:
        print("build OK" if ok else "BUILD FAILED\n" + out)
    return ok, out


def run_c(*args, timeout: int = 300) -> str:
    """Run the compiled binary and return its stdout.

    Exit code 2 means one of the four C exercises is still a stub, so it is re-raised as
    NotImplementedError — the grader then reports TODO instead of an error.
    """
    ok, out = build(verbose=False)
    if not ok:
        raise RuntimeError("the C build failed; run build() to see the compiler output\n" + out)
    proc = subprocess.run([str(LESSON_DIR / BIN), *args],
                          cwd=LESSON_DIR, capture_output=True, text=True, timeout=timeout)
    if proc.returncode == 2:
        raise NotImplementedError(proc.stderr.strip() or "a C exercise is still a stub")
    if proc.returncode != 0:
        raise RuntimeError(f"{BIN} {' '.join(args)} exited {proc.returncode}\n"
                           f"{proc.stderr.strip()}")
    return proc.stdout


def cached(*args) -> str:
    """run_c, but each distinct command line is executed once per session."""
    if args not in _CACHE:
        _CACHE[args] = run_c(*args)
    return _CACHE[args]


def metrics(text: str) -> dict:
    """Parse the binary's `@ key=value` report lines into a dict of floats."""
    return {k: float(v) for k, v in
            (line[2:].split("=", 1) for line in text.splitlines() if line.startswith("@ "))}


def rows(text: str, tag: str) -> list:
    """Parse the binary's `<tag> v1 v2 ...` lines into a list of lists of floats."""
    return [[float(v) for v in line.split()[1:]]
            for line in text.splitlines() if line.startswith(tag + " ")]


def make_test():
    """Run the C self-test — what `make test` runs. Returns (exit code, combined output).

    The binary is built with make and then invoked directly, so its OWN exit code survives:
    make reports any failed recipe as 2, which would make a real failure (the binary's 1)
    indistinguishable from an unfinished stub (the binary's 2).
    """
    ok, out = build(verbose=False)
    if not ok:
        return 1, out
    proc = subprocess.run([str(LESSON_DIR / BIN), "selftest"],
                          cwd=LESSON_DIR, capture_output=True, text=True, timeout=600)
    return proc.returncode, (proc.stdout + proc.stderr).strip()


_FAILURES = []


def _try(fn, *a, **k):
    """Run a public check, print what it says, and remember whether it passed."""
    try:
        fn(*a, **k)
    except NotImplementedError as e:
        print(f"  TODO {fn.__name__}: {e}")
        _FAILURES.append(fn.__name__)
    except AssertionError as e:
        print(f"  FAIL {fn.__name__}: {e}")
        _FAILURES.append(fn.__name__)
    else:
        print(f"  ok   {fn.__name__}")


if _IS_MAIN:
    build()

## 1. The network is the constraint

A model that will not fit the plant network does not get deployed, whatever its ROC curve
says. So this module starts from the network and builds the model to fit it, rather than
fitting the model first and discovering the network afterwards.

The gateway in this lesson sits on the plant side of a firewall. It has one 40 KiB heap and
no swap. It reads a polled fieldbus and publishes one 16-bit register per machine per hour.
The link to the historian goes down. And the path out is a one-way data diode: a device
that NIST describes as one that "cannot be programmed to allow data to flow in both
directions because the hardware is incapable" (`claims.yaml`), so nothing ever comes back.

Every constraint above is a number. Ask the binary what they are, rather than being told.

In [ ]:
if _IS_MAIN:
    _f = metrics(cached("facts"))
    _hours_held = _f["ring_capacity"] / _f["n_machines"]
    print(f"  fleet            {_f['n_machines']:.0f} machines x {_f['n_hours']:.0f} hours "
          f"= {_f['n_frames']:.0f} polls")
    print(f"  one frame        {_f['frame_bytes']:.0f} bytes, carrying "
          f"{_f['burst_len']:.0f} registers")
    print(f"  the capture      {_f['capture_bytes']:.0f} bytes of recorded wire")
    print(f"  gateway heap     {_f['arena_bytes']:.0f} bytes, and there is no other memory")
    print(f"  buffer           {_f['ring_capacity']:.0f} records of "
          f"{_f['record_bytes']:.0f} bytes = {_f['ring_capacity'] * _f['record_bytes']:.0f} "
          f"bytes, {100 * _f['ring_capacity'] * _f['record_bytes'] / _f['arena_bytes']:.1f}% "
          f"of the heap")
    print(f"  which buys you   {_hours_held:.1f} hours of outage before a reading is refused")
    print(f"  the wire         Q{16 - _f['wire_frac']:.0f}.{_f['wire_frac']:.0f} in one "
          f"16-bit register: a resolution of {1 / _f['wire_scale']:.6f} health units")
    print(f"\n  {DATA_NOTE}")

## 2. The capture, byte for byte

Nothing below opens a socket. The "wire" is a recorded capture — the bytes a protocol
analyser left behind — synthesised at start-up from a fixed seed and replayed out of a
static buffer. The buffer is deliberately **not** inside the gateway's arena: it stands in
for the wire, and the gateway never holds more than one frame of it.

Its frames are Modbus RTU register-read responses: unit address, function code, byte count,
registers, then a two-byte CRC whose low-order byte goes first. The layout, the function
code and the 0xA001 polynomial all come from the Modbus Organization's own specifications
(`claims.yaml`); they are not invented for this lesson.

The notebook builds the same bytes in numpy. If the two ever disagree, every comparison
later in this lesson is comparing two different inputs and means nothing, so that is the
first thing checked.

In [ ]:
_CRC_TABLE = None


def _crc_table() -> np.ndarray:
    """The 256-entry table for CRC-16 with the reversed polynomial 0xA001. GIVEN."""
    global _CRC_TABLE
    if _CRC_TABLE is None:
        t = np.zeros(256, dtype=np.uint16)
        for b in range(256):
            crc = b
            for _ in range(8):
                crc = (crc >> 1) ^ 0xA001 if (crc & 1) else (crc >> 1)
            t[b] = crc
        _CRC_TABLE = t
    return _CRC_TABLE


def crc16_rows(block: np.ndarray) -> np.ndarray:
    """CRC-16/MODBUS over every row of a (frames, bytes) uint8 array. GIVEN."""
    t = _crc_table()
    crc = np.full(block.shape[0], 0xFFFF, dtype=np.uint16)
    for j in range(block.shape[1]):
        idx = (crc ^ block[:, j].astype(np.uint16)) & np.uint16(0xFF)
        crc = (crc >> np.uint16(8)) ^ t[idx]
    return crc


def _mix(seed: int, k: np.ndarray) -> np.ndarray:
    """The counter-based stream lesson.c draws from. GIVEN; identical in both languages."""
    z = np.uint64(seed) + (k + np.uint64(1)) * np.uint64(GAMMA)
    z = (z ^ (z >> np.uint64(30))) * np.uint64(0xBF58476D1CE4E5B9)
    z = (z ^ (z >> np.uint64(27))) * np.uint64(0x94D049BB133111EB)
    return z ^ (z >> np.uint64(31))


def baseline_counts(m: np.ndarray) -> np.ndarray:
    """The commissioned baseline of each machine, in raw counts. GIVEN.

    Set once, at commissioning, and stored in the gateway's configuration — which is why the
    gateway and this notebook hold the SAME baseline, and the only differences left between
    their answers are the arithmetic and the wire.
    """
    return 8192 + 64 * np.asarray(m, dtype=np.int64)


def generate_counts() -> np.ndarray:
    """The raw ADC counts behind every frame, as (N_FRAMES, BURST_LEN) uint16. GIVEN.

    Read it: 40 machines run healthy, 24 sit close to the alarm line, and 16 degrade from a
    staggered onset. Every step is integer arithmetic, so lesson.c reproduces it exactly.
    """
    idx = np.arange(N_FRAMES, dtype=np.int64)
    h, m = idx // N_MACHINES, idx % N_MACHINES
    u = _mix(SEED, idx.astype(np.uint64))
    ppm = np.zeros(N_FRAMES, dtype=np.int64)
    healthy = m < N_HEALTHY
    marginal = (m >= N_HEALTHY) & (m < N_HEALTHY + N_MARGINAL)
    degrading = m >= N_HEALTHY + N_MARGINAL
    ppm[healthy] = 1_000_000 + (u[healthy] % np.uint64(120001)).astype(np.int64) - 60_000
    ppm[marginal] = 1_473_200 + (u[marginal] % np.uint64(80001)).astype(np.int64) - 40_000
    d = m[degrading] - (N_HEALTHY + N_MARGINAL)
    onset, slope = 24 + 6 * d, 8_000 + 400 * d
    ramp = np.where(h[degrading] > onset, (h[degrading] - onset) * slope, 0)
    ppm[degrading] = 1_000_000 + ramp + (u[degrading] % np.uint64(40001)).astype(np.int64) - 20_000
    level = baseline_counts(m) * ppm // 1_000_000
    j = (idx[:, None] * BURST_LEN + np.arange(BURST_LEN, dtype=np.int64)[None, :])
    jitter = (_mix(SEED2, j.astype(np.uint64)) % np.uint64(129)).astype(np.int64) - 64
    return np.clip(level[:, None] + jitter, 0, 65535).astype(np.uint16)


def build_capture() -> np.ndarray:
    """The recorded capture, as (N_FRAMES, FRAME_BYTES) uint8. GIVEN.

    Frame layout: address, function code, byte count, big-endian registers, CRC-16 low byte
    first — a Modbus RTU response frame (`claims.yaml`). Every 997th frame carries a flipped
    bit. The protocol carries a checksum precisely because frames arrive damaged, and a
    gateway that trusts every frame it is handed publishes noise as if it were a reading.
    """
    if "capture" not in _CACHE:
        counts = generate_counts()
        cap = np.zeros((N_FRAMES, FRAME_BYTES), dtype=np.uint8)
        cap[:, 0] = (np.arange(N_FRAMES) % N_MACHINES + 1).astype(np.uint8)
        cap[:, 1] = FUNC_READ_INPUT
        cap[:, 2] = 2 * BURST_LEN
        cap[:, 3:3 + 2 * BURST_LEN:2] = (counts >> np.uint16(8)).astype(np.uint8)
        cap[:, 4:4 + 2 * BURST_LEN:2] = (counts & np.uint16(0xFF)).astype(np.uint8)
        crc = crc16_rows(cap[:, :FRAME_BYTES - 2])
        cap[:, FRAME_BYTES - 2] = (crc & np.uint16(0xFF)).astype(np.uint8)
        cap[:, FRAME_BYTES - 1] = (crc >> np.uint16(8)).astype(np.uint8)
        bad = (np.arange(N_FRAMES) % CORRUPT_STRIDE) == CORRUPT_PHASE
        cap[bad, 7] ^= np.uint8(0x40)
        _CACHE["capture"] = cap
    return _CACHE["capture"]


def _check_capture_is_identical_in_c_and_python() -> None:
    """If the two sides do not hold the same bytes, nothing below is evidence."""
    cap = build_capture()
    text = cached("capture", "--dump", "64")
    m = metrics(text)
    flat = cap.reshape(-1).astype(np.uint64)
    want_sum = int(flat.sum())
    # uint64 arithmetic wraps mod 2^64, which is what the C accumulator does too.
    want_wsum = int((np.arange(1, flat.size + 1, dtype=np.uint64) * flat).sum())
    assert int(m["capture_bytes"]) == cap.size, (
        f"lesson.c built {int(m['capture_bytes'])} bytes of capture and numpy built "
        f"{cap.size}; the frame layout or the frame count differs between them")
    assert int(m["capture_sum"]) == want_sum and int(m["capture_wsum"]) == want_wsum, (
        f"the two captures are the same length but not the same bytes: C reports "
        f"sum={int(m['capture_sum'])} wsum={int(m['capture_wsum'])}, numpy computes "
        f"{want_sum} and {want_wsum}. The generator, the seed, the byte order or the CRC "
        "differs between the two sides.")
    dumped = [l.split()[2] for l in text.splitlines() if l.startswith("raw ")]
    for i, hexline in enumerate(dumped):
        want = cap[i].tobytes().hex()
        assert hexline == want, (
            f"frame {i} differs: C emitted {hexline[:24]}... and numpy {want[:24]}...")
    print(f"    {len(dumped)} frames compared byte for byte, and all "
          f"{cap.size:,} bytes agree on both checksums")
    print(f"    {int((np.arange(N_FRAMES) % CORRUPT_STRIDE == CORRUPT_PHASE).sum())} of the "
          f"{N_FRAMES:,} frames carry a flipped bit")


if _IS_MAIN:
    _try(_check_capture_is_identical_in_c_and_python)

## 3. Exercise 1 — `gw_alloc()` in `lesson.c`

The gateway has 40 KiB and no swap. It takes what it needs at start-up, calls `gw_freeze()`
and then runs for a year. `malloc` is not available to it: the four names are `#define`d to
symbols that do not exist, so a call to any of them fails at the **link** step with a name
that tells you why.

Fill in `gw_alloc` in `lesson.c`. `make test` is your feedback loop and the cell below runs
it; every case in it is hand-workable.

In [ ]:
def _check_gw_alloc() -> None:
    m = metrics(cached("arena"))
    assert m["started"] == 1, (
        "the gateway could not take its memory out of the arena at start-up — gw_alloc "
        "returned NULL for one of the four blocks")
    assert m["alloc_calls"] == 4, (
        f"start-up makes exactly four allocations and {m['alloc_calls']:.0f} succeeded; a "
        "refusal here means the budget is too small or the cursor is being advanced by "
        "requests that failed")
    assert m["post_freeze_null"] == 1 and m["alloc_after_freeze"] == 1, (
        "an allocation attempted after gw_freeze() must return NULL and be counted in "
        f"g_alloc_after_freeze; you returned "
        f"{'NULL' if m['post_freeze_null'] else 'a pointer'} and the counter reads "
        f"{m['alloc_after_freeze']:.0f}")
    print(f"    start-up took {m['arena_used']:.0f} of {m['arena_bytes']:.0f} bytes "
          f"({100 * m['arena_used'] / m['arena_bytes']:.1f}%), leaving "
          f"{m['arena_free']:.0f}")
    print(f"    allocations after the door closed: {m['alloc_after_freeze']:.0f} attempted, "
          f"{m['alloc_calls']:.0f} granted in total — all of them before gw_freeze()")


if _IS_MAIN:
    _code, _out = make_test()
    print(_out)
    print(f"exit code {_code}  (0 = all pass, 2 = something is still a stub, 1 = a failure)\n")
    _try(_check_gw_alloc)

## 4. Exercise 2 — `frame_decode()` in `lesson.c`

Parse one frame off the recorded capture: address, function code, byte count, big-endian
registers, CRC-16 low byte first. Three things the exercise insists on, and all three are
things that bite on real plant buses.

**Length before content.** A frame shorter than its own header is not a frame with a bad
checksum, it is not a frame.
**Big-endian registers.** The specification says the register data are "packed as two
bytes per register" and that "the first byte contains the high order bits and the second
contains the low order bits" (`claims.yaml`). Get that backwards and `0x1234` reads as
`0x3412` — 4660 becomes 13330 — and every health index downstream of it is wrong without
a single error being raised.
**A rejected frame writes nothing.** A half-filled struct is worse than an empty one,
because the caller cannot tell which half is real.

In [ ]:
def _check_frame_decode() -> None:
    cap = build_capture()
    text = cached("health")
    m = metrics(text)
    bad = {int(r[0]) for r in rows(text, "bad")}
    want_bad = {i for i in range(N_FRAMES) if i % CORRUPT_STRIDE == CORRUPT_PHASE}
    assert bad == want_bad, (
        f"the frames your decoder rejected are not the ones the capture corrupts: it "
        f"rejected {len(bad)} and {len(want_bad)} carry a flipped bit. Extra rejections "
        "usually mean the checksum is being read big-endian; missing ones mean it is not "
        "being checked at all.")
    codes = {int(r[2]) for r in rows(text, "bad")}
    assert codes == {-4}, (
        f"every corrupt frame in this capture fails its CHECKSUM, so the code is FD_CRC "
        f"(-4); you returned {sorted(codes)}")
    good = rows(text, "h")
    assert len(good) == N_FRAMES - len(want_bad), (
        f"{len(good)} frames decoded and {N_FRAMES - len(want_bad)} should have")
    machines = {int(r[1]) for r in good}
    assert machines == set(range(N_MACHINES)), (
        "the decoded unit addresses do not cover every machine; the address is byte 0 and "
        "the machine index is that minus one")
    print(f"    {len(good):,} frames decoded, {len(bad)} rejected on their checksum")
    print(f"    the plant lost {100 * len(bad) / N_FRAMES:.2f}% of its readings to line "
          f"noise, before the model saw anything")
    print(f"    first rejected frame: index {min(bad)}, machine {min(bad) % N_MACHINES}, "
          f"hour {min(bad) // N_MACHINES}")
    print(f"    byte 7 of that frame on the wire: 0x{cap[min(bad)][7]:02x}")


if _IS_MAIN:
    _try(_check_frame_decode)

## 5. Exercise 3 — `q_health()` in `lesson.c`

The inference path, on the device. No FPU, no `sqrt`, no rounding mode to argue about:
a raw count is `Q16.16`, the squares accumulate in `int64_t`, the square root is computed
digit by digit, and every step **truncates towards zero**.

Two overflows are waiting: the accumulator, and then `rms_q << 16` one line further down.
Neither warns, and both are in the self-test. The cell below prints how far past an
`int32_t` a full-scale burst carries each of them, before it checks anything.

This is the number that will go on the wire, so get it exact before asking what it costs.

In [ ]:
def _check_q_health() -> None:
    text = cached("health")
    good = rows(text, "h")
    hq = np.array([r[3] for r in good], dtype=np.int64)
    assert (hq > 0).all(), (
        "at least one reading came back as -1 or 0; q_health returns -1 only for n <= 0 or "
        "a baseline <= 0, and every frame in this capture carries 32 registers")
    m = np.array([int(r[1]) for r in good])
    counts = generate_counts()
    keep = np.array([int(r[0]) for r in good])
    x = counts[keep].astype(np.float64) / float(1 << SAMPLE_SHIFT)
    ref = np.sqrt((x * x).mean(axis=1)) / (baseline_counts(m) / float(1 << SAMPLE_SHIFT))
    internal = hq / float(1 << Q_FRAC)
    gap = np.abs(internal - ref)
    assert gap.max() < 4.0 / (1 << Q_FRAC), (
        f"the gateway's health index is up to {gap.max():.6g} away from the float64 answer, "
        f"which is more than a few steps of Q16.16 ({1 / (1 << Q_FRAC):.6g}). That is not "
        "truncation, that is a different computation — check the accumulator width and that "
        "the shift into the numerator happens in int64_t.")
    assert (internal <= ref + 1e-12).all(), (
        "some readings came back ABOVE the float64 answer. Every step of the fixed-point "
        "path truncates towards zero, so the gateway's answer can only ever be lower; a "
        "reading above it means something is rounding.")
    print(f"    {len(good):,} readings, every one at or below the float64 answer")
    print(f"    largest arithmetic gap {gap.max():.3e}, mean {gap.mean():.3e}")
    print(f"    one step of Q16.16 is {1 / (1 << Q_FRAC):.3e}")


if _IS_MAIN:
    # GIVEN, and printed rather than asserted: how far past an int32_t each of the two
    # overflows carries a full-scale burst. Both run whether or not the exercise is done.
    _full = (65535 << SAMPLE_SHIFT)
    _acc = BURST_LEN * _full * _full
    _shifted = _full << Q_FRAC
    _i32 = 2 ** 31 - 1
    print(f"    {BURST_LEN} squares of a full-scale sample sum to {_acc:.3e}, which is "
          f"{_acc / _i32:,.0f}x the largest value an int32_t holds")
    print(f"    and rms_q << {Q_FRAC} for that burst is {_shifted:.3e}, "
          f"{_shifted / _i32:,.0f}x — the same bug one line further down")
    _try(_check_q_health)

## 6. Exercise 4 — `float_health()`, in Python

Now the same inference path in the language the model was developed in: float64, numpy,
one line of `sqrt`. This is the half everybody writes first and the half nobody deploys.

It is here so that the disagreement in the next section is a measurement between two
implementations you wrote, rather than an assertion about fixed point in general.

In [ ]:
def float_health(counts: np.ndarray, baseline: np.ndarray) -> np.ndarray:
    """The reference inference path, in float64: rms of the burst over the baseline.

    A raw ADC count c is the engineering value c / 2**SAMPLE_SHIFT. The health index of a
    burst is the root mean square of those values divided by the machine's commissioned
    baseline, which is the same definition module 1 used — a machine sitting on its baseline
    reads 1.0.

    Arguments:
      counts    an integer array whose LAST axis is the burst, e.g. (n_readings, BURST_LEN)
      baseline  the commissioned baseline in ENGINEERING units, broadcastable against
                counts.shape[:-1]

    Requirements, all graded:
      * compute in float64 throughout, however the caller's array is typed;
      * reduce over the LAST axis only;
      * raise ValueError if any baseline is <= 0, and if the burst axis is empty.

    Returns: a float64 array of shape counts.shape[:-1] holding the health index.

    Worked example: a burst of 32 counts all equal to 2048, against a baseline of 8.0. Each
    engineering value is 2048 / 256 = 8.0, the root mean square of 32 copies of 8.0 is 8.0,
    and 8.0 / 8.0 is 1.0.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement float_health")


def _check_float_health() -> None:
    flat = float_health(np.full((1, BURST_LEN), 2048), np.array([8.0]))
    assert flat.shape == (1,) and abs(float(flat[0]) - 1.0) < 1e-15, (
        f"the docstring's worked example must give 1.0 with shape (1,); you returned "
        f"{flat!r}")
    twice = float_health(np.full((1, BURST_LEN), 4096), np.array([8.0]))
    assert abs(float(twice[0]) - 2.0) < 1e-15, (
        f"twice the baseline is a health index of 2.0; you returned {float(twice[0])!r}")
    ramp = np.arange(BURST_LEN, dtype=np.int64)[None, :]
    want = math.sqrt(float((ramp.astype(np.float64) ** 2).mean())) / (1 << SAMPLE_SHIFT) / 1.0
    assert abs(float(float_health(ramp, np.array([1.0]))[0]) - want) < 1e-12, (
        "on a ramp your answer differs from the root MEAN square; check that you are not "
        "taking the mean of the values and squaring it, or summing without dividing by n")
    for bad, why in ((np.array([0.0]), "a baseline of 0"), (np.array([-1.0]), "a negative "
                                                                              "baseline")):
        try:
            float_health(np.full((1, BURST_LEN), 2048), bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"{why} must raise ValueError, not return a number")
    counts = generate_counts()
    m = np.arange(N_FRAMES) % N_MACHINES
    h = float_health(counts, baseline_counts(m) / float(1 << SAMPLE_SHIFT))
    assert h.shape == (N_FRAMES,), f"on the whole capture the shape must be ({N_FRAMES},)"
    print(f"    health over the week: min {h.min():.4f}, median {np.median(h):.4f}, "
          f"max {h.max():.4f}")
    print(f"    readings at or above the threshold of {THRESHOLD}: "
          f"{int((h >= THRESHOLD).sum()):,} of {h.size:,}")


if _IS_MAIN:
    _try(_check_float_health)

## 7. Exercise 5 — `agreement_report()`: the disagreement IS the deployment risk

Two implementations of one model. The gap between them is not a curiosity; it is the thing
that will be discovered at 3 a.m. by somebody who does not have either source file.

There are two gaps and they have different causes, so measure them separately:

* the **arithmetic** gap, between float64 and the gateway's internal `Q16.16` — truncation
  in the accumulator, the square root and the divide;
* the **wire** gap, between float64 and the `Q8.8` value that fits in one 16-bit register.

One of those is a property of your code. The other is a property of somebody else's
register map, and it is the larger one.

In [ ]:
def agreement_report(h_float: np.ndarray, h_wire: np.ndarray, threshold: float) -> dict:
    """Quantify the disagreement between two implementations of one inference path.

    Arguments:
      h_float    the float64 health index, one value per reading
      h_wire     the same readings as the deployed path reports them, already converted back
                 from the register to engineering units
      threshold  the alarm threshold the plant applies to BOTH

    Both alarm decisions use `>= threshold` on the same number, because that is what the
    plant does: it reads a register, scales it and compares it with the threshold it was
    given. Do not compare the wire values against a different, "corrected" threshold — the
    whole point is to find out what the threshold the plant was given actually means once it
    reaches the wire.

    `effective_threshold` is that meaning, and it is the only value here you derive rather
    than measure: the smallest value representable on a Q8.8 wire that is >= threshold, i.e.
    ceil(threshold * WIRE_SCALE) / WIRE_SCALE. A wire value clears the threshold if and only
    if it is at or above that, so the effective threshold is what the plant is really using.

    Requirements, all graded:
      * raise ValueError if the two arrays do not have the same shape;
      * report the LARGEST and the MEAN absolute difference, and the flat index of the worst
        reading, so the worst case can be looked at rather than described;
      * count the two directions of disagreement SEPARATELY. Which direction they fall in is
        the finding: a deployment that only ever misses alarms and a deployment that only
        ever adds them are different risks with different owners;
      * return plain Python floats and ints, not numpy scalars, so the report survives being
        written to a file.

    Returns: a dict with keys max_abs_error, mean_abs_error, worst_index,
    effective_threshold, n_float_only, n_wire_only and n_disagreements.

    Worked example: h_float = [1.50, 1.60], h_wire = [1.25, 1.60], threshold = 1.55. The
    absolute differences are 0.25 and 0.0, so max_abs_error is 0.25 and worst_index is 0.
    Both rows agree on the alarm (no, yes), so all three counts are 0. The effective
    threshold is ceil(1.55 * 256) / 256 = 397 / 256 = 1.55078125.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement agreement_report")


def deployed_health() -> tuple:
    """Pull the gateway's answers off the capture. GIVEN.

    Returns (index, machine, hour, internal, wire): the flat frame index of every reading
    that decoded, the machine and hour it belongs to, the gateway's internal Q16.16 health
    as a float, and the Q8.8 register value converted back to engineering units.
    """
    good = rows(cached("health"), "h")
    arr = np.array(good, dtype=np.float64)
    return (arr[:, 0].astype(np.int64), arr[:, 1].astype(np.int64),
            arr[:, 2].astype(np.int64), arr[:, 3] / float(1 << Q_FRAC),
            arr[:, 4] / float(WIRE_SCALE))


def _check_agreement_report() -> None:
    toy = agreement_report(np.array([1.50, 1.60]), np.array([1.25, 1.60]), 1.55)
    assert abs(toy["max_abs_error"] - 0.25) < 1e-15 and toy["worst_index"] == 0, (
        f"the docstring's worked example must give max_abs_error 0.25 at index 0; you "
        f"returned {toy['max_abs_error']!r} at {toy['worst_index']!r}")
    assert toy["n_disagreements"] == 0, (
        f"both rows agree on the alarm, so n_disagreements is 0; you returned "
        f"{toy['n_disagreements']!r}")
    assert toy["effective_threshold"] == 397 / 256, (
        f"the smallest Q8.8 value at or above 1.55 is 397/256 = {397 / 256!r}; you returned "
        f"{toy['effective_threshold']!r}")
    assert isinstance(toy["max_abs_error"], float) and isinstance(toy["n_float_only"], int), (
        "return plain Python floats and ints; a numpy scalar does not survive being written "
        "to a report")
    try:
        agreement_report(np.zeros(3), np.zeros(4), 1.0)
    except ValueError:
        pass
    else:
        raise AssertionError("mismatched shapes must raise ValueError")

    idx, mach, hour, internal, wire = deployed_health()
    counts = generate_counts()
    hf = float_health(counts[idx], baseline_counts(mach) / float(1 << SAMPLE_SHIFT))
    arith = agreement_report(hf, internal, THRESHOLD)
    onwire = agreement_report(hf, wire, THRESHOLD)
    assert onwire["max_abs_error"] > arith["max_abs_error"], (
        "the wire gap came out no bigger than the arithmetic gap, which cannot be right: "
        "Q8.8 throws away eight of the sixteen fractional bits the gateway computed with")
    ratio = onwire["max_abs_error"] / arith["max_abs_error"]
    print(f"    arithmetic gap   max {arith['max_abs_error']:.3e}  "
          f"mean {arith['mean_abs_error']:.3e}   <- your fixed-point path")
    print(f"    wire gap         max {onwire['max_abs_error']:.3e}  "
          f"mean {onwire['mean_abs_error']:.3e}   <- their register map")
    print(f"    the register map costs {ratio:,.0f}x what your arithmetic does")
    print(f"    one step of the wire is {1 / WIRE_SCALE:.6f}; the largest wire gap is "
          f"{onwire['max_abs_error'] * WIRE_SCALE:.6f} of a step — a whole step of "
          "truncation, plus the arithmetic underneath it")


if _IS_MAIN:
    _try(_check_agreement_report)

## 8. A threshold that does not fit the wire

Now the part that changes decisions rather than digits. A 16-bit `Q8.8` register can hold
`k / 256` and nothing in between, so the plant's comparison `value >= 1.4732` is really
`k >= ceil(1.4732 * 256)`. The threshold you were handed is not the threshold you deployed.

Two things follow, and the cell below measures both. The shift is in **one direction**,
because every step of the fixed-point path truncates downwards — so a deployment like this
can only ever go quiet, never noisy. And a threshold that lands exactly on a representable
value costs nothing at all, which is a deployment decision somebody is allowed to make.

In [ ]:
def _check_the_threshold_moved() -> None:
    idx, mach, hour, internal, wire = deployed_health()
    counts = generate_counts()
    hf = float_health(counts[idx], baseline_counts(mach) / float(1 << SAMPLE_SHIFT))
    rep = agreement_report(hf, wire, THRESHOLD)
    eff = rep["effective_threshold"]
    assert rep["n_wire_only"] == 0, (
        f"{rep['n_wire_only']} readings alarm on the wire and not in float64. Every step of "
        "the deployed path truncates towards zero, so the wire value can never exceed the "
        "float64 one — if it does, something in q_health or health_to_wire is rounding.")
    assert rep["n_float_only"] > 0, (
        "no reading changed its alarm decision at all, on a threshold that is not "
        "representable in Q8.8. Check that the wire values really are the 16-bit register "
        "and not the internal Q16.16 value.")
    below = math.floor(THRESHOLD * WIRE_SCALE) / WIRE_SCALE
    print(f"    the threshold you were handed        {THRESHOLD}")
    print(f"    the nearest wire values              {below} and {eff}")
    print(f"    the threshold you actually deployed  {eff}  "
          f"(+{100 * (eff - THRESHOLD) / THRESHOLD:.3f}%)")
    print(f"    readings that alarm in float64 and not on the wire: {rep['n_float_only']:,} "
          f"of {hf.size:,}")
    print(f"    readings that alarm on the wire and not in float64: {rep['n_wire_only']:,}")

    # And what that does to the hour an engineer is told about.
    first_f, first_w = {}, {}
    for machine, hr, a, b in zip(mach, hour, hf >= THRESHOLD, wire >= THRESHOLD):
        if a and machine not in first_f:
            first_f[int(machine)] = int(hr)
        if b and machine not in first_w:
            first_w[int(machine)] = int(hr)
    moved = {k: first_w.get(k) for k in first_f if first_w.get(k) != first_f[k]}
    late = [first_w[k] - first_f[k] for k in moved if k in first_w]
    print(f"    machines whose FIRST alarm moved: {len(moved)} of {len(first_f)} that "
          f"alarmed at all")
    if late:
        print(f"    and when it moved it moved late, by {min(late)} to {max(late)} hours "
              f"(median {int(np.median(late))})")
    print(f"    snapping the threshold up to {eff} at commissioning costs nothing and")
    print("    makes the deployed number the same on both sides of the firewall.")


if _IS_MAIN:
    _try(_check_the_threshold_moved)

## 9. Exercise 6 — store and forward, in `lesson.c`

The link goes down. The machines do not. `sf_push`, `sf_take` and `sf_ack` have to give the
historian **exactly once** delivery across an outage, and the two halves of that live in
two different places:

* **nothing lost** — a full buffer refuses a reading and counts the refusal, rather than
  overwriting the oldest unacknowledged record. A refusal is a number you can put in a
  report; an overwrite is a reading that never existed. And a refused reading must not
  consume a sequence number, so the delivered sequence stays contiguous.
* **nothing duplicated** — `sf_take` is a peek, not a pop, because the link is allowed to
  deliver a batch and lose the acknowledgement coming back. The gateway will then offer the
  same records again, and the only thing standing between that and a double-counted
  reading is the historian's record of the highest sequence number it has stored.

Four scenarios, one capture.

In [ ]:
_SCENARIOS = ("clean", "outage", "lostack", "overflow")


def _check_store_and_forward() -> None:
    seen = {s: metrics(cached("forward", "--scenario", s)) for s in _SCENARIOS}
    for name, m in seen.items():
        pushed = int(m["pushed"])
        assert int(m["accepted"]) == pushed, (
            f"[{name}] {pushed} readings entered the buffer and {int(m['accepted'])} "
            "reached the historian. Every reading that was accepted at the door has to "
            "arrive: check that sf_ack only drops records at or below the sequence number "
            "it was given, and that sf_take starts at head.")
        want_sum = pushed * (pushed + 1) // 2
        assert int(m["sum_seq"]) == want_sum and int(m["max_seq"]) == pushed, (
            f"[{name}] the delivered sequence numbers are not 1..{pushed}: they sum to "
            f"{int(m['sum_seq'])} instead of {want_sum} and top out at "
            f"{int(m['max_seq'])}. A gap means a reading was lost in transit; a refused "
            "push that consumed a sequence number looks exactly like one.")
        assert int(m["held_at_end"]) == 0, (
            f"[{name}] {int(m['held_at_end'])} records are still in the buffer at the end "
            "of the week, with the link up and everything acknowledged")
        assert m["alloc_after_freeze"] == 0, (
            f"[{name}] the gateway allocated after start-up, which it must never do")

    assert int(seen["clean"]["duplicates"]) == 0 and int(seen["outage"]["duplicates"]) == 0, (
        "no acknowledgement is lost in the clean or outage scenarios, so the historian "
        "should never see a record twice")
    assert int(seen["lostack"]["duplicates"]) > 0, (
        "the lostack scenario drops four hours of acknowledgements, so the gateway must "
        "re-offer those records and the historian must reject them. Your historian saw no "
        "duplicates at all, which means sf_take is removing the records it hands out.")
    assert int(seen["overflow"]["refused"]) > 0, (
        "a hundred-hour outage does not fit in the buffer, so some readings must be refused")
    assert int(seen["clean"]["refused"]) == 0 and int(seen["outage"]["refused"]) == 0, (
        "the outage is shorter than the buffer holds, so nothing should have been refused")

    hours = seen["outage"]["ring_capacity"] / N_MACHINES
    print(f"    scenario    pushed  refused  accepted  dup  peak backlog")
    for name in _SCENARIOS:
        m = seen[name]
        print(f"    {name:<10s}{int(m['pushed']):>8,}{int(m['refused']):>9,}"
              f"{int(m['accepted']):>10,}{int(m['duplicates']):>5,}"
              f"{int(m['peak_backlog']):>14,}")
    print(f"    the buffer holds {int(seen['outage']['ring_capacity']):,} records = "
          f"{hours:.1f} hours of this plant")
    print(f"    the 36-hour outage peaked at {int(seen['outage']['peak_backlog']):,} records "
          f"({100 * seen['outage']['peak_backlog'] / seen['outage']['ring_capacity']:.1f}% "
          f"of the buffer) and lost nothing")
    print(f"    the 100-hour one refused {int(seen['overflow']['refused']):,} readings and "
          f"still delivered the {int(seen['overflow']['accepted']):,} it accepted, exactly "
          "once each")
    print(f"    arena high-water mark: {int(seen['overflow']['arena_peak']):,} of "
          f"{GW_ARENA_BYTES:,} bytes, and "
          f"{int(seen['overflow']['alloc_after_freeze'])} allocations after start-up")


if _IS_MAIN:
    _try(_check_store_and_forward)

## 10. Exercise 7 — what the diode costs you

The outbound path here is a **one-way data diode**, which NIST's OT security guide defines
as "a network appliance or device that allows data to travel only in one direction"
(`claims.yaml`). Light goes out, nothing comes back, and that is the point. It also means no
work order, no confirmed failure and no correction ever reaches the model side. Module 8 has
to detect drift with **no labels at all**, for ever.

So measure what is left. A supervised drift signal — how well the detector ranks confirmed
outcomes — needs labels and has none. An unsupervised one compares the distribution of the
published feature against its commissioning window and needs nothing but the feature.
Implement the second.

In [ ]:
def psi(reference: np.ndarray, current: np.ndarray, n_bins: int = 10,
        floor: float = 1e-6) -> float:
    """Population stability index of `current` against `reference`.

    The bins come from the REFERENCE — the commissioning window, the thing you are drifting
    away from. Deriving them from the current window instead is the classic mistake: the
    bins then move with the data and the statistic stops being able to see the move.

    The recipe:
      * edges = n_bins + 1 equally spaced points from reference.min() to reference.max(),
        with the two outer edges replaced by -inf and +inf so nothing in `current` can fall
        outside the bins;
      * p = the proportion of `reference` in each bin, q = the proportion of `current`;
      * floor both at `floor` before dividing, because an empty bin in either sample would
        otherwise give an infinite or undefined term — a real window does have empty bins;
      * psi = sum over bins of (q - p) * ln(q / p).

    Requirements, all graded:
      * raise ValueError if either sample is empty, if n_bins < 2, or if the reference has no
        spread at all (min == max), because there are then no bins to build;
      * use natural logarithms;
      * the result is never negative, and is exactly 0.0 when the two samples fall in the
        bins in identical proportions.

    Returns: the PSI as a plain Python float.

    Worked example: psi(x, x) is 0.0 for any x with some spread — every bin has q == p, so
    every term is 0 * ln(1). Move every value in `current` above reference.max() and all of
    it lands in the open top bin, which is what the infinite outer edges are for.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement psi")


def supervised_drift(scores: np.ndarray, labels: np.ndarray) -> float:
    """How well the detector ranks confirmed outcomes. GIVEN — and it needs labels.

    Returns the rank AUC, or nan when there is nothing to rank. Across a diode there are no
    labels at all, so this returns nan every hour of every week, for ever.
    """
    s, y = np.asarray(scores, dtype=np.float64), np.asarray(labels, dtype=np.int64)
    if s.size == 0 or y.size == 0 or y.min() == y.max():
        return float("nan")
    order = np.argsort(s, kind="mergesort")
    ranks = np.empty(s.size, dtype=np.float64)
    ranks[order] = np.arange(1, s.size + 1, dtype=np.float64)
    npos, nneg = int((y == 1).sum()), int((y == 0).sum())
    return float((ranks[y == 1].sum() - npos * (npos + 1) / 2) / (npos * nneg))


def _check_psi() -> None:
    rng = np.random.default_rng(SEED)
    a = rng.normal(0.0, 1.0, 4000)
    assert psi(a, a) == 0.0, (
        f"the same window against itself is exactly 0.0; you returned {psi(a, a)!r}")
    b = a + 0.75
    moved = psi(a, b)
    assert moved > 0.1, (
        f"a shift of three quarters of a standard deviation should be plainly visible; you "
        f"returned {moved!r}. If it is near zero, the bins are probably being derived from "
        "the current window instead of the reference.")
    assert psi(a, b) != psi(b, a), (
        "PSI is not symmetric here, because the bins come from the reference. Getting the "
        "same number both ways means the edges are not built from `reference`.")
    far = psi(a, a + 50.0)
    assert math.isfinite(far) and far > 0, (
        f"a window that lands entirely outside the reference range must give a finite, "
        f"positive answer; you returned {far!r}. The outer edges are -inf and +inf and the "
        "empty bins are floored.")
    for args, why in (((np.array([]), a), "an empty reference"),
                      ((a, np.array([])), "an empty current window"),
                      ((np.ones(10), a), "a reference with no spread")):
        try:
            psi(*args)
        except ValueError:
            pass
        else:
            raise AssertionError(f"{why} must raise ValueError")
    try:
        psi(a, b, n_bins=1)
    except ValueError:
        pass
    else:
        raise AssertionError("n_bins < 2 must raise ValueError")
    gapped = np.concatenate([rng.normal(0.0, 0.2, 500), rng.normal(10.0, 0.2, 500)])
    middle = rng.normal(5.0, 0.2, 500)
    hole = psi(gapped, middle)
    assert math.isfinite(hole) and hole > 0, (
        f"a reference with a gap in the middle has EMPTY bins, and a current window that "
        f"lands in them divides by a proportion of zero; you returned {hole!r}. Floor BOTH "
        "proportions, not just the current one.")
    print(f"    same window against itself   {psi(a, a):.6f}")
    print(f"    shifted by 0.75 sigma        {moved:.6f}")
    print(f"    moved clean off the range    {far:.6f}")
    print(f"    into a hole in the reference {hole:.6f}")


if _IS_MAIN:
    _try(_check_psi)

## 11. The diode, priced

Here is the whole argument in one table. The same week, the same published register, and
the one thing that changes is whether anything is allowed back through the firewall.

A sensor recalibration is module 8's first kind of drift: the feature moves and the machine
does not. Below, one is stipulated — the raw counts are rescaled by 101/100, a figure chosen
because it is small rather than because it is typical — and both detectors are asked about
it.

In [ ]:
def _check_the_diode_leaves_only_one_signal() -> None:
    idx, mach, hour, internal, wire = deployed_health()
    counts = generate_counts()
    healthy = mach < N_HEALTHY
    ref = wire[healthy & (hour < 84)]
    same = wire[healthy & (hour >= 84)]

    # The same hours, with the raw counts rescaled by 101/100. Integer arithmetic, so the
    # recalibration is exactly reproducible; the size of it is stipulated, not measured.
    recal_counts = (counts[idx].astype(np.int64) * 101) // 100
    recal = float_health(recal_counts, baseline_counts(mach) / float(1 << SAMPLE_SHIFT))
    recal = np.floor(recal * WIRE_SCALE) / WIRE_SCALE
    recal = recal[healthy & (hour >= 84)]

    quiet, drifted = psi(ref, same), psi(ref, recal)
    assert drifted > quiet, (
        f"a 1% recalibration should move the feature distribution further than the second "
        f"half of an undisturbed week does: you measured {drifted:.6f} against "
        f"{quiet:.6f}")

    # The labels. On a bidirectional link every alarm becomes a work order and the fitter's
    # finding comes back; across a diode none of them do, by construction. The finding here
    # is ground truth the generator knows and the plant does not: a machine is genuinely
    # faulty once it is past its own degradation onset.
    alarm = wire >= THRESHOLD
    d = mach - (N_HEALTHY + N_MARGINAL)
    faulty = (d >= 0) & (hour > 24 + 6 * np.maximum(d, 0))
    labels_bidirectional, labels_diode = int(alarm.sum()), 0
    sup_bi = supervised_drift(wire[alarm], faulty[alarm].astype(np.int64))
    sup_diode = supervised_drift(wire[:0], np.array([], dtype=np.int64))
    assert math.isnan(sup_diode), (
        "with no labels the supervised signal is not a small number, it is no number at all")
    assert 0.0 < sup_bi < 1.0, (
        f"the supervised signal came out at {sup_bi!r}; an AUC of exactly 1.0 means the "
        "labels were derived from the scores, which measures nothing")

    print("    link                      labels back   supervised   unsupervised (PSI)")
    print(f"    bidirectional             {labels_bidirectional:>11,}   {sup_bi:>10.4f}   "
          f"{drifted:>18.6f}")
    print(f"    one-way diode             {labels_diode:>11,}   {'nan':>10s}   "
          f"{drifted:>18.6f}")
    print()
    print(f"    PSI, undisturbed second half of the week : {quiet:.6f}")
    print(f"    PSI, the same hours after a 1% recalibration: {drifted:.6f} "
          f"({drifted / max(quiet, 1e-12):,.0f}x)")
    print(f"    The diode costs you {labels_bidirectional:,} labels a week and zero of the")
    print("    unsupervised signal. That is why module 8 is built on the second column.")


if _IS_MAIN:
    _try(_check_the_diode_leaves_only_one_signal)

## 12. Common mistakes

**A binary built before you moved the checkout.** `make` decides a binary is up to date by
comparing timestamps, and a binary that arrived with a copied or moved directory is *newer*
than the `.c` beside it — so `make` will not rebuild it and you will be grading yesterday's
answers. Worse, a compiled lesson that links a library out of a virtualenv has that
library's absolute path baked in by the linker, and after a move the loader cannot find it
at all. **Run `make clean` after moving or copying your checkout, or after rebuilding the
virtualenv.** This lesson links nothing but the system maths library, so only the
stale-timestamp half can bite here; the repository's execution gate deletes compiled
artefacts before grading any C lesson for exactly that reason.

**Sizing the buffer in records instead of in hours.** A record count is not a number
anybody can argue with. The hours-of-plant figure section 1 printed, set beside the longest
outage the site actually had last year, is.

**A ring buffer that overwrites when it is full.** It is the default behaviour of nearly
every ring buffer ever written, and it turns a countable refusal into a silent hole.

**Treating delivery as exactly-once because the code looks like it.** The link loses
acknowledgements. Delivery is at-least-once and the sequence number is what makes it
exactly-once — at the far end, not here.

**Testing the model against the model.** Run the cell.

In [ ]:
def _demonstrate_common_mistakes() -> None:
    idx, mach, hour, internal, wire = deployed_health()
    counts = generate_counts()
    hf = float_health(counts[idx], baseline_counts(mach) / float(1 << SAMPLE_SHIFT))
    print(f"    the model against itself: {agreement_report(hf, hf, THRESHOLD)['n_disagreements']} "
          "disagreements, which is what a validation that never leaves the laptop measures")
    print(f"    the model against what was deployed: "
          f"{agreement_report(hf, wire, THRESHOLD)['n_disagreements']:,}")
    worst = agreement_report(hf, wire, THRESHOLD)["worst_index"]
    print(f"    the worst single reading is machine {int(mach[worst])} at hour "
          f"{int(hour[worst])}: float64 {hf[worst]:.6f}, wire {wire[worst]:.6f}")
    m = metrics(cached("forward", "--scenario", "overflow"))
    print(f"    a buffer that overwrote instead of refusing would have reported "
          f"{int(m['accepted']):,} readings delivered and said nothing about the "
          f"{int(m['refused']):,} it dropped")
    print(f"    32-bit accumulation of 32 full-scale squares would need "
          f"{32 * (65535 * 256) ** 2 / 2 ** 31:,.0f}x the range an int32_t has")


if _IS_MAIN:
    _try(_demonstrate_common_mistakes)

## 13. Self-check

**1.** Your gateway publishes the health index as a `Q8.8` register and the threshold that
arrived from upstream is 1.4732. What is the plant actually comparing against?
(a) 1.4732 · (b) the largest representable value below it · (c) the smallest representable
value at or above it · (d) whichever of the two is nearer

**2.** The link has been down for longer than the buffer holds — section 1 printed how
many hours of this plant that is. What must `sf_push` do with the next reading?
(a) overwrite the oldest unacknowledged record · (b) refuse it and count the refusal ·
(c) allocate more room · (d) drop every second record to make space

**3.** The historian stores a batch and the acknowledgement is lost on the way back, so the
gateway offers the same records again. What stops them being stored twice?
(a) the frame's CRC · (b) `sf_take` removing the records it hands out · (c) the ring's
capacity · (d) the historian comparing each record's sequence number against the highest
it has already stored

**4.** Across a one-way diode, which of these can module 8 still compute?
(a) the detector's AUC against confirmed work orders · (b) the measured false-alarm rate ·
(c) the distribution of the published feature against its commissioning window ·
(d) precision at the chosen operating point

In [ ]:
SELF_CHECK = {1: "?", 2: "?", 3: "?", 4: "?"}   # <- put a, b, c or d in each


# The marker holds a salted digest of each answer, not the answer. It still tells you
# immediately which questions are wrong and which section settles each one, but reading this
# cell does not hand you the four letters.
_MARK = {
    1: ("08f2aa17ce1b23e701dbcaa7615437dca6b58c637c45fbb10603504bb3345abb", "section 8"),
    2: ("01451309899193d66e4c628a1502ca4569b5f6c5f7768ee1f59f52b61bf97ca5", "section 9"),
    3: ("f07c2345df71a754fad76b0dd70ac8c77f357fa100dafd825e0cfeca4badb3c0", "section 9"),
    4: ("c1a71a48854329f0fce80c24142c5ea09a02b36c4e4505f5740918108a9ec565", "section 11"),
}


def _check_self_check(answers: dict = None) -> None:
    """Mark the four multiple-choice answers, naming the section that settles each."""
    answers = SELF_CHECK if answers is None else answers
    wrong = [q for q, (want, _) in _MARK.items()
             if hashlib.sha256(
                 f"P03-L07:{q}:{str(answers.get(q, '?')).strip().lower()}".encode()
             ).hexdigest() != want]
    assert not wrong, (
        "questions " + ", ".join(str(q) for q in wrong) + " are still wrong. Look again at "
        + "; ".join(f"q{q}: {_MARK[q][1]}" for q in wrong) + ".")
    print("self-check: all four right")


if _IS_MAIN:
    _try(_check_self_check)

## 14. What you built

One model, two implementations, and a measured gap between them that is mostly somebody
else's register map rather than your arithmetic. A buffer that says what it dropped instead
of dropping it quietly. And a week of readings that crossed a firewall exactly once each.

The threshold module 6 handed you is not the threshold you deployed until you snap it onto
the wire's grid and write down which way you rounded. That sentence, with the two numbers
beside it, is the deliverable.

Module 8 picks up the last column of section 11: the plant can see its own feature
distribution and nothing else, so drift detection, retraining triggers and the alarm audit
all have to work without a single label coming back.

In [ ]:
if _IS_MAIN:
    if _FAILURES:
        print(f"\n{len(_FAILURES)} check(s) still failing: {', '.join(_FAILURES)}")
        sys.exit(1)
    print("\nall public checks green — now run:  python tools/grade.py <this lesson>")